In [1]:
import os
import pandas as pd
import json
from PIL import Image
from tqdm import tqdm

# 1. 설정값 (5060Ti VRAM 방어를 위한 최적 사이즈)
# Qwen-VL은 해상도를 유동적으로 받지만, 768px 정도면 성능 저하 없이 메모리를 획기적으로 아낍니다.
MAX_SIZE = 768 
OUTPUT_BASE_DIR = "resized_images"

def resize_images_and_update_jsonl(csv_file, input_jsonl=None, output_jsonl=None):
    if not os.path.exists(csv_file):
        return
        
    df = pd.read_csv(csv_file)
    # VLM 트랙 데이터만 필터링
    vlm_df = df[df['class'].fillna('VLM') != 'YOLO']
    
    print(f"\n🚀 [{csv_file}] VLM 이미지 {len(vlm_df)}장 리사이징 시작...")
    
    # 2. 이미지 리사이징 및 새 폴더에 저장
    for idx, row in tqdm(vlm_df.iterrows(), total=len(vlm_df)):
        img_path = str(row['path']) # 예: 'train/train_0001.jpg'
        
        if not os.path.exists(img_path):
            continue
            
        # 새 저장 경로 생성 (예: 'resized_images/train/train_0001.jpg')
        dest_path = os.path.join(OUTPUT_BASE_DIR, img_path)
        os.makedirs(os.path.dirname(dest_path), exist_ok=True)
        
        # 이미지 열기 -> 비율 유지하며 리사이징 -> 압축 저장
        with Image.open(img_path) as img:
            # RGBA(투명도) 등이 섞인 이미지가 있을 수 있으므로 RGB로 안전하게 변환
            img = img.convert('RGB') 
            # thumbnail 함수는 비율(Aspect Ratio)을 깨지 않고 긴 축을 MAX_SIZE에 맞춥니다.
            img.thumbnail((MAX_SIZE, MAX_SIZE), Image.Resampling.LANCZOS)
            # 화질 85% 정도로 압축하여 디스크 용량과 로딩 속도 최적화
            img.save(dest_path, format="JPEG", quality=85)
            
    # 3. JSONL 파일이 있다면 경로를 새 폴더(resized_images)로 업데이트
    if input_jsonl and os.path.exists(input_jsonl):
        print(f"🔄 {input_jsonl} 내부의 이미지 경로를 업데이트합니다...")
        updated_data = []
        with open(input_jsonl, 'r', encoding='utf-8') as f:
            for line in f:
                record = json.loads(line)
                # 프롬프트 안의 "<image>train/..." 부분을 "<image>resized_images/train/..." 로 교체
                record['messages'][0]['content'] = record['messages'][0]['content'].replace(
                    "<image>train/", f"<image>{OUTPUT_BASE_DIR}/train/"
                ).replace(
                    "<image>dev/", f"<image>{OUTPUT_BASE_DIR}/dev/"
                )
                updated_data.append(record)
                
        # 덮어쓰거나 새 이름으로 저장 (여기서는 덮어씁니다)
        with open(output_jsonl, 'w', encoding='utf-8') as f:
            for item in updated_data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        print(f"✅ {output_jsonl} 경로 업데이트 완료!")

# 실행부
# Train 셋 처리 및 jsonl 경로 변경
resize_images_and_update_jsonl('train_class.csv', 'train_vlm.jsonl', 'train_vlm_resized.jsonl')
# Dev 셋 처리 및 jsonl 경로 변경
resize_images_and_update_jsonl('dev_class.csv', 'dev_vlm.jsonl', 'dev_vlm_resized.jsonl')
# Test 셋은 학습용 jsonl이 없으므로 이미지만 1회성으로 리사이징 해둡니다.
resize_images_and_update_jsonl('test_class.csv')

print("\n🎉 VLM용 이미지 리사이징 및 프롬프트 경로 매핑이 모두 완료되었습니다!")


🚀 [train_class.csv] VLM 이미지 3326장 리사이징 시작...


100% 3326/3326 [02:08<00:00, 25.94it/s]


🔄 train_vlm.jsonl 내부의 이미지 경로를 업데이트합니다...
✅ train_vlm_resized.jsonl 경로 업데이트 완료!

🚀 [dev_class.csv] VLM 이미지 1269장 리사이징 시작...


100% 1269/1269 [00:49<00:00, 25.78it/s]


🔄 dev_vlm.jsonl 내부의 이미지 경로를 업데이트합니다...
✅ dev_vlm_resized.jsonl 경로 업데이트 완료!

🚀 [test_class.csv] VLM 이미지 3365장 리사이징 시작...


100% 3365/3365 [02:07<00:00, 26.39it/s]


🎉 VLM용 이미지 리사이징 및 프롬프트 경로 매핑이 모두 완료되었습니다!
